
# (CLAUDE REFORMATED FOR GOOGLE COLLAB) Cas12a CNN

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

### Setup steps
1. Run Cell 1 to mount your Google Drive
2. Make sure your Drive has this folder structure:
```
MyDrive/cas12a_project/
    data/training_sets/raw_data/
        Kim_2018_Train.csv
        Kim_2018_Test.csv
    model/scripts/
        feature_engineering.py
```
3. Run cells top to bottom

In [1]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# !! CHANGE THIS to match your actual Drive folder name
PROJECT = "/content/drive/MyDrive/cas12a_project"

import os
print("Project folder exists:", os.path.exists(PROJECT))
print("Contents:", os.listdir(PROJECT))

Mounted at /content/drive
Project folder exists: True
Contents: ['.DS_Store', 'data', '.git', 'model', 'cnn_model']


In [2]:
# ── Cell 2: Check GPU ────────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

CUDA available: True
GPU: Tesla T4


In [3]:
# ── Cell 3: Imports & Paths ──────────────────────────────────────────────────
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader, Dataset

# point to your feature_engineering.py on Drive
sys.path.insert(0, os.path.join(PROJECT, "model", "scripts"))
import feature_engineering

# paths
DATASET_PATH = os.path.join(PROJECT, "data", "training_sets", "raw_data")
TRAIN_FILE   = "Kim_2018_Train.csv"
TEST_FILE    = "Kim_2018_Test.csv"
WEIGHTS_DIR  = os.path.join(PROJECT, "cnn_model", "weights")
OUTPUT_DIR   = os.path.join(PROJECT, "cnn_model", "results")
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,  exist_ok=True)

TARGET_COL = "Indel frequency"
INP_COL    = "Context Sequence"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
# ── Cell 4: Config — change your training settings here ─────────────────────
CFG = {
    "epochs":       50,
    "batch_size":   32,
    "lr":           1e-3,
    "dropout":      0.3,
    "weight_decay": 1e-4,
    "patience":     15,     # early stopping: epochs without improvement before stopping a fold
    "cv_folds":     2,
    "handcrafted":  True,  # set True to inject GC/Tm/dG features at the FC layer
    "run_cv":       True,   # set False to skip cross-validation and go straight to final training
}
print("Config:", CFG)

Config: {'epochs': 100, 'batch_size': 32, 'lr': 0.001, 'dropout': 0.3, 'weight_decay': 0.0001, 'patience': 15, 'cv_folds': 5, 'handcrafted': False, 'run_cv': True}


In [5]:
# ── Cell 5: Data utilities ───────────────────────────────────────────────────

def one_hot_encode(seq: str) -> np.ndarray:
    """Convert a DNA string to a (4, 34) float32 one-hot array."""
    base_i = {"A": 0, "C": 1, "G": 2, "T": 3}
    seq = seq.upper().strip()
    arr = np.zeros((4, 34), dtype=np.float32)
    for i, base in enumerate(seq[:34]):
        idx = base_i.get(base)
        if idx is not None:
            arr[idx, i] = 1.0
    return arr


class GRNADataset(Dataset):
    def __init__(self, sequences, targets, handcrafted_features=None):
        # convert everything to tensors once upfront — avoids per-batch overhead
        self.seqs = torch.tensor(np.stack([one_hot_encode(s) for s in sequences]))
        self.y    = torch.tensor(targets.astype(np.float32))
        self.hc   = torch.tensor(handcrafted_features) if handcrafted_features is not None else None

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, i):
        if self.hc is not None:
            return self.seqs[i], self.hc[i], self.y[i]
        return self.seqs[i], self.y[i]


def load_data(path):
    df = pd.read_csv(path).dropna(subset=[TARGET_COL])
    return df.reset_index(drop=True)


def normalize_target(train_df, test_df):
    """Z-score normalise target. Reuses saved stats if they exist."""
    mean_path = os.path.join(WEIGHTS_DIR, "target_mean.npy")
    std_path  = os.path.join(WEIGHTS_DIR, "target_std.npy")
    if os.path.exists(mean_path) and os.path.exists(std_path):
        mean = float(np.load(mean_path))
        std  = float(np.load(std_path))
        print(f"[normalize] Loaded saved stats: mean={mean:.4f}, std={std:.4f}")
    else:
        mean = train_df[TARGET_COL].mean()
        std  = train_df[TARGET_COL].std()
        np.save(mean_path, mean)
        np.save(std_path,  std)
        print(f"[normalize] Computed stats: mean={mean:.4f}, std={std:.4f}")
    train_df = train_df.copy()
    test_df  = test_df.copy()
    train_df[TARGET_COL] = (train_df[TARGET_COL] - mean) / std
    test_df[TARGET_COL]  = (test_df[TARGET_COL]  - mean) / std
    return train_df, test_df, mean, std


def build_hc_features(sequences, mean=None, std=None):
    """
    Build scalar hand-crafted features using feature_engineering pipeline.
    Pass mean/std from training call when calling on test set so both
    are normalised on the same scale.
    """
    SCALAR_FEATURES = [
        'gc_content', 
        'tm', 
        'nn_dg', 
        'self_comp', 
        'homopolymer_count', 
        'mono_A', 'mono_C', 'mono_G', 'mono_T', 
        'di_repeats', 
        'pos_gc', 
        'pam_t_count', 'pam_is_tttv', 
        'spacer_gc', 
        'seed_gc', 
        'seed_a_count', 
        'cleavage_gc',
        'spacer_tm', 
        'spacer_nn_dg', 
        'full_self_comp', 
        'upstream_A', 'upstream_C', 'upstream_G', 'upstream_T',
    ]
    df   = feature_engineering.build_features(sequences)
    cols = [c for c in SCALAR_FEATURES if c in df.columns]
    arr  = df[cols].values.astype(np.float32)
    arr  = np.nan_to_num(arr)
    if mean is None:
        mean = arr.mean(axis=0, keepdims=True)
        std  = arr.std(axis=0,  keepdims=True) + 1e-8
    return (arr - mean) / std, len(cols), mean, std


def evaluate(y_true, y_pred, prefix=""):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    pr,_ = pearsonr(y_true, y_pred)
    sr,_ = spearmanr(y_true, y_pred)
    print(f"  {prefix}RMSE={rmse:.4f}  MAE={mae:.4f}  Pearson r={pr:.4f}  Spearman rho={sr:.4f}")
    return {"rmse": rmse, "mae": mae, "pearson": pr, "spearman": sr}

print("Data utilities defined.")

Data utilities defined.


In [6]:
# ── Cell 6: Model definition ─────────────────────────────────────────────────

class ResConvBlock(nn.Module):
    """Two conv layers with a residual skip connection."""
    def __init__(self, in_ch, out_ch, kernel):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(in_ch,  out_ch, kernel, padding=pad)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=pad)
        self.bn2   = nn.BatchNorm1d(out_ch)
        # project input channels to match output if needed for residual addition
        self.proj  = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        residual = self.proj(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual)


class AttentionPool(nn.Module):
    """Learn a weight per sequence position, then take a weighted sum."""
    def __init__(self, channels):
        super().__init__()
        self.attn = nn.Conv1d(channels, 1, 1)

    def forward(self, x):                        # x: (B, C, 34)
        w = torch.softmax(self.attn(x), dim=-1)  # (B, 1, 34) — weights sum to 1
        return (x * w).sum(dim=-1)               # (B, C) — weighted sum over positions


class CNN(nn.Module):
    """
    Multi-scale CNN for Cas12a indel frequency prediction.

    Input:  (B, 4, 34)  one-hot encoded sequence
    Output: (B,)        normalised indel frequency
    """
    def __init__(self, hc_dim: int = 0, dropout: float = 0.3):
        super().__init__()

        # three parallel towers scanning at different scales
        self.tower3 = nn.Sequential(ResConvBlock(4, 64, 3), ResConvBlock(64, 64, 3))
        self.tower5 = nn.Sequential(ResConvBlock(4, 64, 5), ResConvBlock(64, 64, 5))
        self.tower7 = nn.Sequential(ResConvBlock(4, 64, 7), ResConvBlock(64, 64, 7))

        # merge all three towers: 64*3=192 channels → 128
        self.merge = ResConvBlock(192, 128, 3)

        # collapse 34 positions → single vector
        self.pool = AttentionPool(128)

        # fully connected head
        fc_in = 128 + hc_dim  # hc_dim=0 if not using handcrafted features
        self.fc = nn.Sequential(
            # nn.Linear(fc_in, 128),
            # nn.BatchNorm1d(128),
            # nn.ReLU(),
            # nn.Dropout(dropout),
            # nn.Linear(128, 64),
            # nn.ReLU(),
            # nn.Dropout(dropout / 2),
            # nn.Linear(64, 1),
            nn.Linear(fc_in, 64),    # was 128→128→64, now just 128→64
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq, x_hc=None):
        t3 = self.tower3(x_seq)
        t5 = self.tower5(x_seq)
        t7 = self.tower7(x_seq)
        x  = torch.cat([t3, t5, t7], dim=1)  # (B, 192, 34)
        x  = self.merge(x)                    # (B, 128, 34)
        x  = self.pool(x)                     # (B, 128)
        if x_hc is not None:
            x = torch.cat([x, x_hc], dim=1)
        return self.fc(x).squeeze(1)          # (B,)

print("Model defined.")
# quick sanity check
dummy = torch.zeros(2, 4, 34)
out   = CNN()(dummy)
print(f"Sanity check — input: {dummy.shape}  output: {out.shape}  (expected (2,))")

Model defined.
Sanity check — input: torch.Size([2, 4, 34])  output: torch.Size([2])  (expected (2,))


In [7]:
# ── Cell 7: Training loop functions ─────────────────────────────────────────

def train_epoch(model, loader, optimiser, use_hc):
    model.train()
    total_loss = 0.0
    for batch in loader:
        if use_hc:
            x_seq, x_hc, y = [b.to(device) for b in batch]
            pred = model(x_seq, x_hc)
        else:
            x_seq, y = [b.to(device) for b in batch]
            pred = model(x_seq)
        loss = F.mse_loss(pred, y)
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_epoch(model, loader, use_hc):
    model.eval()
    preds, targets = [], []
    for batch in loader:
        if use_hc:
            x_seq, x_hc, y = [b.to(device) for b in batch]
            pred = model(x_seq, x_hc)
        else:
            x_seq, y = [b.to(device) for b in batch]
            pred = model(x_seq)
        preds.append(pred.cpu().numpy())
        targets.append(y.cpu().numpy())
    return np.concatenate(preds), np.concatenate(targets)

print("Training functions defined.")

Training functions defined.


In [8]:
# ── Cell 8: Load data ────────────────────────────────────────────────────────
train_df = load_data(os.path.join(DATASET_PATH, TRAIN_FILE))
test_df  = load_data(os.path.join(DATASET_PATH, TEST_FILE))
print(f"Train: {len(train_df)} sequences | Test: {len(test_df)} sequences")

train_df, test_df, t_mean, t_std = normalize_target(train_df, test_df)

train_seqs = train_df[INP_COL].tolist()
test_seqs  = test_df[INP_COL].tolist()
y_train    = train_df[TARGET_COL].values.astype(np.float32)
y_test     = test_df[TARGET_COL].values.astype(np.float32)

# optional hand-crafted features
hc_train = hc_test = None
hc_dim = 0
if CFG["handcrafted"]:
    print("[features] Building hand-crafted features...")
    hc_train, hc_dim, hc_mean, hc_std = build_hc_features(train_seqs)
    hc_test,  _,      _,       _      = build_hc_features(test_seqs, mean=hc_mean, std=hc_std)
    if CFG["handcrafted"]:
        np.save(os.path.join(WEIGHTS_DIR, "hc_mean.npy"), hc_mean)
        np.save(os.path.join(WEIGHTS_DIR, "hc_std.npy"),  hc_std)
    print(f"[features] Hand-crafted dim = {hc_dim}")

train_ds = GRNADataset(train_seqs, y_train, hc_train)
test_ds  = GRNADataset(test_seqs,  y_test,  hc_test)
print("Datasets ready.")

Train: 17813 sequences | Test: 150 sequences
[normalize] Loaded saved stats: mean=40.7937, std=31.8233
Datasets ready.


In [9]:
# ── Cell 9: Cross-validation (skip by setting run_cv=False in CFG) ───────────
if CFG["run_cv"]:
    print(f"[cv] {CFG['cv_folds']}-fold cross-validation")
    kf = KFold(n_splits=CFG["cv_folds"], shuffle=True, random_state=42)
    cv_results = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(range(len(train_ds))), 1):
        tr_loader  = DataLoader(train_ds, batch_size=CFG["batch_size"],
                                sampler=torch.utils.data.SubsetRandomSampler(tr_idx),
                                num_workers=2, pin_memory=True)
        val_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                                sampler=torch.utils.data.SubsetRandomSampler(val_idx),
                                num_workers=2, pin_memory=True)

        model = CNN(hc_dim=hc_dim, dropout=CFG["dropout"]).to(device)
        opt   = torch.optim.Adam(model.parameters(), lr=CFG["lr"],
                                 weight_decay=CFG["weight_decay"])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])

        best_val, patience_ctr = np.inf, 0
        for epoch in range(1, CFG["epochs"] + 1):
            train_epoch(model, tr_loader, opt, CFG["handcrafted"])
            val_pred, val_true = eval_epoch(model, val_loader, CFG["handcrafted"])
            val_loss = mean_squared_error(val_true, val_pred)
            sched.step()
            if val_loss < best_val:
                best_val, patience_ctr = val_loss, 0
            else:
                patience_ctr += 1
            if patience_ctr >= CFG["patience"]:
                print(f"  Fold {fold}: early stop at epoch {epoch}")
                break

        print(f"Fold {fold} results:", end="")
        m = evaluate(val_true, val_pred)
        cv_results.append(m)

    print("\n[cv] Aggregate across folds:")
    for key in ["rmse", "mae", "pearson", "spearman"]:
        vals = [m[key] for m in cv_results]
        print(f"  {key}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

    pd.DataFrame(cv_results).to_csv(
        os.path.join(OUTPUT_DIR, "cnn_cv_metrics.csv"), index=False)
    print(f"CV results saved to {OUTPUT_DIR}/cnn_cv_metrics.csv")
else:
    print("[cv] Skipped (run_cv=False in CFG)")

[cv] 5-fold cross-validation
  Fold 1: early stop at epoch 25
Fold 1 results:  RMSE=0.7719  MAE=0.5726  Pearson r=0.6814  Spearman rho=0.6917


KeyboardInterrupt: 

In [ ]:
# ── Cell 10: Final model training ────────────────────────────────────────────
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"],
                          num_workers=2, pin_memory=True)

model = CNN(hc_dim=hc_dim, dropout=CFG["dropout"]).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=CFG["lr"],
                         weight_decay=CFG["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])

ckpt_path     = os.path.join(WEIGHTS_DIR, "cnn_best.pt")
best_test_rmse = np.inf

print(f"[train] Training for up to {CFG['epochs']} epochs on {device}...")

import json
cnn_config = {"hc_dim": hc_dim, "dropout": CFG["dropout"]}
with open(os.path.join(WEIGHTS_DIR, "cnn_config.json"), "w") as f:
    json.dump(cnn_config, f)
print(f"Saved cnn_config: {cnn_config}")

for epoch in range(1, CFG["epochs"] + 1):
    tr_loss = train_epoch(model, train_loader, opt, CFG["handcrafted"])
    y_pred_norm, y_true_norm = eval_epoch(model, test_loader, CFG["handcrafted"])
    rmse = np.sqrt(mean_squared_error(y_true_norm, y_pred_norm))
    sched.step()

    if epoch % 10 == 0 or epoch == 1:
        pr, _ = pearsonr(y_true_norm, y_pred_norm)
        print(f"  epoch {epoch:3d}  train_loss={tr_loss:.4f}  test_RMSE={rmse:.4f}  Pearson={pr:.4f}")

    if rmse < best_test_rmse:
        best_test_rmse = rmse
        torch.save(model.state_dict(), ckpt_path)

print(f"\nBest test RMSE: {best_test_rmse:.4f}")
print(f"Checkpoint saved: {ckpt_path}")

In [ ]:
# ── Cell 11: Final evaluation with best checkpoint ───────────────────────────
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
y_pred_norm, y_true_norm = eval_epoch(model, test_loader, CFG["handcrafted"])

print("[results] Final test performance:")
print("  Normalised scale:")
evaluate(y_true_norm, y_pred_norm)
print("  Percentage scale (denormalised):")
evaluate(y_true_norm * t_std + t_mean, y_pred_norm * t_std + t_mean)

pred_df = pd.DataFrame({
    "context_sequence": test_seqs,
    "y_true_pct":  y_true_norm * t_std + t_mean,
    "y_pred_pct":  y_pred_norm * t_std + t_mean,
})
out_csv = os.path.join(OUTPUT_DIR, "cnn_predictions.csv")
pred_df.to_csv(out_csv, index=False)
print(f"Predictions saved: {out_csv}")